In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("file_date", "2024-12-16", "File Date")
file_date = dbutils.widgets.get("file_date")

In [0]:
from pyspark.sql.functions import col, lit,  concat_ws, trim
from delta.tables import DeltaTable

In [0]:
persons_df = (
    spark.table("databricks_ws.bronze.persons")
    .filter(col("file_date") == file_date)
    )

In [0]:
persons_selected_df = (
    persons_df
    .select(
        col("personId").cast("int"),
        col("personName.forename").alias("forename"),
        col("personName.surname").alias("surname"),
        col("file_date")
    )
)

In [0]:
persons_with_columns_df = (
    add_ingestion_date(persons_selected_df)
    .withColumnRenamed("personId", "person_id")
    .withColumn("environment", lit("Production"))
    .withColumn("name", concat_ws(" ", trim(col("forename")), trim(col("surname"))))
)

In [0]:
persons_final_df = (
    persons_with_columns_df
    .drop("forename")
    .drop("surname")
)

In [0]:
table_name = "databricks_ws.silver.persons"
if spark.catalog.tableExists(table_name):

    deltaTable = DeltaTable.forName(spark, table_name)

    deltaTable.alias("tgt") \
        .merge(
            persons_final_df.alias("src"),
            "tgt.person_id = src.person_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    persons_final_df.write \
        .format("delta") \
        .saveAsTable(table_name)